In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data import load_region_df
from src.soc_engine import BatterySpec, SoCEngine, DispatchAction
from src.forecaster import (
    BATTERY_LZ_COL,
    train_houston_price_model,
    generate_forecast_range,
)
import src.dam_bidder
from src.dam_bidder import (
    optimise_dam_bid,
    settle_dam_bid,
    run_dam_campaign,
)
from src.forecast_error import (
    run_perfect_foresight,
    run_perfect_foresight_campaign,
    compute_forecast_error_analysis,
)
import importlib
importlib.reload(src.dam_bidder)
from src.dam_bidder import optimise_dam_bid, settle_dam_bid, run_dam_campaign

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

# ── Setup ──────────────────────────────────────────────────────────────────
print("Loading data and training model...")
df_coast   = load_region_df("coast")
spec       = BatterySpec(
    power_mw=100,
    duration_hours=4,
    charge_eff=0.92,
    discharge_eff=0.92,
    min_soc_pct=10.0,
    max_soc_pct=100.0,
    initial_soc_pct=50.0,
)
test_start = pd.Timestamp("2025-01-01")
sim_start  = pd.Timestamp("2025-06-01")
sim_end    = pd.Timestamp("2025-06-14")

model_result = train_houston_price_model(df_coast, test_start)
forecast_df  = generate_forecast_range(model_result, df_coast, sim_start, sim_end)
campaign_df  = run_dam_campaign(forecast_df, df_coast, spec)

print(f"Campaign: {len(campaign_df)} days\n")
print("-" * 60)

# ══════════════════════════════════════════════════════════════
# BLOCK 1 — SoC COLUMN PRESENCE
# ══════════════════════════════════════════════════════════════
print("\n── Block 1: SoC Column Presence ──")
check(
    "starting_soc_pct" in campaign_df.columns,
    "B1.1 — starting_soc_pct column present in campaign output",
    "B1.1 — starting_soc_pct MISSING from campaign output",
)
check(
    "ending_soc_pct" in campaign_df.columns,
    "B1.2 — ending_soc_pct column present in campaign output",
    "B1.2 — ending_soc_pct MISSING from campaign output",
)
check(
    "soc_violations" in campaign_df.columns,
    "B1.3 — soc_violations column present in campaign output",
    "B1.3 — soc_violations MISSING from campaign output",
)

# ══════════════════════════════════════════════════════════════
# BLOCK 2 — INITIAL STATE
# ══════════════════════════════════════════════════════════════
print("\n── Block 2: Initial State ──")
day1_start = campaign_df.iloc[0]["starting_soc_pct"]
check(
    abs(day1_start - spec.initial_soc_pct) < 0.01,
    f"B2.1 — Day 1 starting SoC = spec.initial_soc_pct ({spec.initial_soc_pct}%): actual={day1_start:.2f}%",
    f"B2.1 — Day 1 starting SoC wrong: expected {spec.initial_soc_pct}%, got {day1_start:.2f}%",
)

# ══════════════════════════════════════════════════════════════
# BLOCK 3 — DAY-TO-DAY CONTINUITY
# ══════════════════════════════════════════════════════════════
print("\n── Block 3: Day-to-Day SoC Continuity ──")
continuity_violations = 0
for i in range(len(campaign_df) - 1):
    day_n_end     = campaign_df.iloc[i]["ending_soc_pct"]
    day_n1_start  = campaign_df.iloc[i + 1]["starting_soc_pct"]
    gap           = abs(day_n_end - day_n1_start)
    if gap > 0.01:
        continuity_violations += 1
        print(
            f"  ❌ Continuity break between day {i+1} and {i+2}: end={day_n_end:.2f}% → start={day_n1_start:.2f}% (gap={gap:.4f}%)"
        )

check(
    continuity_violations == 0,
    f"B3.1 — SoC continuous across all {len(campaign_df)-1} day transitions: no gaps detected",
    f"B3.1 — {continuity_violations} continuity breaks found — engine is being reset between days",
)

print("\n  Day-by-day SoC (first 5 days):")
print("  {:<6} {:<14} {:>10} {:>10} {:>12}".format("Day", "Date", "Start SoC", "End SoC", "Violations"))
print("-" * 56)
for idx in range(min(5, len(campaign_df))):
    row = campaign_df.iloc[idx]
    print(
        f"  {idx+1:<6} {str(row['date']):<14} {row['starting_soc_pct']:>9.2f}% {row['ending_soc_pct']:>9.2f}% {row['soc_violations']:>12}"
    )


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 4 — SOC BOUNDS NEVER VIOLATED
# ══════════════════════════════════════════════════════════════
print("\n── Block 4: SoC Bounds ──")
check(
    (campaign_df["starting_soc_pct"] >= spec.min_soc_pct - 0.01).all(),
    f"B4.1 — all starting SoC >= min ({spec.min_soc_pct}%): min observed={campaign_df['starting_soc_pct'].min():.2f}%",
    f"B4.1 — starting SoC violated min on {(campaign_df['starting_soc_pct'] < spec.min_soc_pct).sum()} days",
)
check(
    (campaign_df["ending_soc_pct"] >= spec.min_soc_pct - 0.01).all(),
    f"B4.2 — all ending SoC >= min ({spec.min_soc_pct}%): min observed={campaign_df['ending_soc_pct'].min():.2f}%",
    f"B4.2 — ending SoC violated min on {(campaign_df['ending_soc_pct'] < spec.min_soc_pct).sum()} days",
)
check(
    (campaign_df["starting_soc_pct"] <= spec.max_soc_pct + 0.01).all(),
    f"B4.3 — all starting SoC <= max ({spec.max_soc_pct}%): max observed={campaign_df['starting_soc_pct'].max():.2f}%",
    f"B4.3 — starting SoC exceeded max on {(campaign_df['starting_soc_pct'] > spec.max_soc_pct).sum()} days",
)
check(
    (campaign_df["ending_soc_pct"] <= spec.max_soc_pct + 0.01).all(),
    f"B4.4 — all ending SoC <= max ({spec.max_soc_pct}%): max observed={campaign_df['ending_soc_pct'].max():.2f}%",
    f"B4.4 — ending SoC exceeded max on {(campaign_df['ending_soc_pct'] > spec.max_soc_pct).sum()} days",
)

# ══════════════════════════════════════════════════════════════
# BLOCK 5 — VIOLATIONS REDUCED (hours the SoC engine had to clip)
# Each "violation" = one hour where requested charge/discharge was clipped by SoC bounds.
# The bid now uses effective_charge_mw / effective_discharge_mw so dispatch matches
# achievable energy — clipping (and violations) go to zero when the fix is applied.
# ══════════════════════════════════════════════════════════════
print("\n── Block 5: Violation Count ──")
total_violations = campaign_df["soc_violations"].sum()
days             = len(campaign_df)
avg_per_day      = total_violations / days if days > 0 else 0

check(
    avg_per_day < 1.0,
    f"B5.1 — avg violations per day < 1.0: {avg_per_day:.2f} ({total_violations} total over {days} days)",
    f"B5.1 — avg violations per day = {avg_per_day:.2f} — still too high",
)
check(
    total_violations < days * 0.5,
    f"B5.2 — total violations ({total_violations}) < 50% of days ({days})",
    f"B5.2 — total violations ({total_violations}) >= 50% of days",
)

zero_violation_days = (campaign_df["soc_violations"] == 0).sum()
check(
    zero_violation_days >= days * 0.7 if days > 0 else True,
    f"B5.3 — {zero_violation_days}/{days} days ({zero_violation_days/days*100:.1f}%) have zero violations",
    f"B5.3 — only {zero_violation_days}/{days} days have zero violations — expected >= 70%",
)

print(f"\n  Violation summary: Total={total_violations}, Days with={days - zero_violation_days}, Days without={zero_violation_days}, Avg/day={avg_per_day:.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 6 — BID FEASIBILITY WITH STARTING SOC
# ══════════════════════════════════════════════════════════════
print("\n── Block 6: Bid Feasibility Given Starting SoC ──")

prices_flat = pd.Series([5.0]*4 + [30.0]*16 + [80.0]*4, index=range(24))

# Test 1 — nearly full battery (95% SoC)
# Headroom = 5% × 400 MWh = 20 MWh
# Full charge = 100MW × 0.92 × 4h = 368 MWh
# Achievable = 20 MWh = 5.4% of full charge
# 5.4% < 50% threshold → charge window should still be skipped
full_soc_mwh   = spec.energy_capacity_mwh * 0.95
headroom       = spec.energy_capacity_mwh * 0.05
full_charge    = spec.power_mw * spec.charge_eff * int(spec.duration_hours)
achievable_pct = headroom / full_charge * 100

bid_full = optimise_dam_bid(
    forecast_prices=prices_flat,
    spec=spec,
    operating_date=pd.Timestamp("2025-06-15"),
    starting_soc_mwh=full_soc_mwh,
)
check(
    len(bid_full.charge_hours) == 0,
    f"B6.1 — nearly full battery (95% SoC, {achievable_pct:.1f}% achievable): charge window skipped (below 50% threshold) charge_hours={bid_full.charge_hours}",
    f"B6.1 — nearly full battery still has charge_hours={bid_full.charge_hours}",
)

# Test 2 — nearly empty battery (12% SoC)
# Available = (12% - 10%) × 400 MWh = 8 MWh
# Full discharge = 100MW × 4h = 400 MWh
# Achievable = 8 MWh = 2% of full discharge
# 2% < 50% threshold → discharge window should be skipped
empty_soc_mwh  = spec.energy_capacity_mwh * 0.12
available      = (0.12 - 0.10) * spec.energy_capacity_mwh
full_discharge = spec.power_mw * int(spec.duration_hours)
avail_pct      = available / full_discharge * 100

bid_empty = optimise_dam_bid(
    forecast_prices=prices_flat,
    spec=spec,
    operating_date=pd.Timestamp("2025-06-15"),
    starting_soc_mwh=empty_soc_mwh,
)
check(
    len(bid_empty.discharge_hours) == 0,
    f"B6.2 — nearly empty battery (12% SoC, {avail_pct:.1f}% achievable): discharge window skipped (below 50% threshold) discharge_hours={bid_empty.discharge_hours}",
    f"B6.2 — nearly empty battery still has discharge_hours={bid_empty.discharge_hours}",
)

# Test 3 — normal SoC (50%)
normal_soc_mwh = spec.energy_capacity_mwh * 0.50

# Diagnostic — print what the optimiser computed for normal SoC
normal_headroom    = spec.max_soc_mwh - normal_soc_mwh
full_chg          = spec.power_mw * spec.charge_eff * int(spec.duration_hours)
achievable_chg    = min(full_chg, normal_headroom)
soc_post_chg      = normal_soc_mwh + achievable_chg
avail_dis         = soc_post_chg - spec.min_soc_mwh
full_dis          = spec.power_mw * int(spec.duration_hours)
achievable_dis    = min(full_dis, avail_dis)

print("\n  B6.3 diagnostics (normal SoC = 50%):")
print(f"    charge_headroom:          {normal_headroom:.1f} MWh")
print(f"    full_charge_energy:       {full_chg:.1f} MWh")
print(f"    achievable_charge:        {achievable_chg:.1f} MWh ({achievable_chg/full_chg*100:.1f}% of full)")
print(f"    min_viable_charge (50%):  {full_chg*0.5:.1f} MWh")
print(f"    charge feasible:          {achievable_chg >= full_chg * 0.5}")
print(f"    soc_after_charge:        {soc_post_chg:.1f} MWh")
print(f"    discharge_available:      {avail_dis:.1f} MWh")
print(f"    achievable_discharge:     {achievable_dis:.1f} MWh ({achievable_dis/full_dis*100:.1f}% of full)")
print(f"    min_viable_discharge(50%): {full_dis*0.5:.1f} MWh")
print(f"    discharge feasible:      {achievable_dis >= full_dis * 0.5}")

bid_normal = optimise_dam_bid(
    forecast_prices=prices_flat,
    spec=spec,
    operating_date=pd.Timestamp("2025-06-15"),
    starting_soc_mwh=normal_soc_mwh,
)
check(
    len(bid_normal.charge_hours) > 0 and len(bid_normal.discharge_hours) > 0,
    f"B6.3 — normal SoC (50%): valid bid produced charge={bid_normal.charge_hours} discharge={bid_normal.discharge_hours}",
    "B6.3 — normal SoC still producing empty bid — check diagnostics above",
)

# B6.4 — boundary: exactly 50% achievable
boundary_headroom = full_chg * 0.50
boundary_soc_mwh  = spec.max_soc_mwh - boundary_headroom

print("\n  B6.4 diagnostics (boundary SoC):")
print(f"    boundary_soc_mwh:         {boundary_soc_mwh:.1f} MWh ({boundary_soc_mwh/spec.energy_capacity_mwh*100:.1f}%)")
print(f"    headroom:                 {boundary_headroom:.1f} MWh")
print(f"    achievable_charge:        {boundary_headroom:.1f} MWh (= exactly 50% of {full_chg:.1f})")
print(f"    threshold (>=):           {full_chg*0.5:.1f} MWh")
print(f"    should pass:              {boundary_headroom >= full_chg * 0.5}")

bid_boundary = optimise_dam_bid(
    forecast_prices=prices_flat,
    spec=spec,
    operating_date=pd.Timestamp("2025-06-15"),
    starting_soc_mwh=boundary_soc_mwh,
)
check(
    len(bid_boundary.charge_hours) > 0,
    f"B6.4 — boundary case (exactly 50% achievable): bid produced charge={bid_boundary.charge_hours}",
    "B6.4 — boundary case rejected — check diagnostic above for threshold comparison direction",
)

# Causal constraint still holds
if len(bid_normal.charge_hours) > 0 and len(bid_normal.discharge_hours) > 0:
    check(
        min(bid_normal.discharge_hours) > max(bid_normal.charge_hours),
        f"B6.5 — causal: discharge starts hour {min(bid_normal.discharge_hours)} > charge ends {max(bid_normal.charge_hours)}",
        "B6.5 — causal constraint violated",
    )


In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 7 — SETTLE_DAM_BID USES EXTERNAL ENGINE
# ══════════════════════════════════════════════════════════════
print("\n── Block 7: External Engine in settle_dam_bid ──")
engine_test   = SoCEngine(spec)
engine_test.soc_mwh = spec.energy_capacity_mwh * 0.50
soc_before    = engine_test.soc_mwh

test_day      = pd.Timestamp("2025-06-01")
test_fcst_day = forecast_df[forecast_df.index.date == test_day.date()]
test_prices   = df_coast[df_coast.index.date == test_day.date()][BATTERY_LZ_COL]

test_prices_series = pd.Series(test_prices.values, index=test_prices.index)
forecast_prices_day = pd.Series(test_fcst_day["forecast_houston_price"].values, index=range(len(test_fcst_day)))

test_bid = optimise_dam_bid(
    forecast_prices=forecast_prices_day,
    spec=spec,
    operating_date=test_day,
    starting_soc_mwh=soc_before,
)
settlement = settle_dam_bid(
    bid=test_bid,
    actual_prices=test_prices_series,
    spec=spec,
    engine=engine_test,
)
soc_after = engine_test.soc_mwh

check(
    abs(soc_after - spec.energy_capacity_mwh * settlement["ending_soc_pct"] / 100) < 0.1,
    f"B7.1 — external engine SoC advanced correctly after settlement: {soc_before/spec.energy_capacity_mwh*100:.1f}% → {soc_after/spec.energy_capacity_mwh*100:.1f}%",
    "B7.1 — engine SoC did not advance as expected",
)
check(
    soc_after != soc_before or (len(test_bid.charge_hours) == 0 and len(test_bid.discharge_hours) == 0),
    "B7.2 — engine state changed after settlement (or idle bid — both valid)",
    "B7.2 — engine state unchanged — settlement may not be using engine",
)

settlement_fresh = settle_dam_bid(
    bid=test_bid,
    actual_prices=test_prices_series,
    spec=spec,
    engine=None,
)
check(
    isinstance(settlement_fresh, dict),
    "B7.3 — settle_dam_bid works with engine=None (standalone mode)",
    "B7.3 — settle_dam_bid failed with engine=None",
)

# ══════════════════════════════════════════════════════════════
# BLOCK 8 — PERFECT FORESIGHT CAMPAIGN CONTINUITY
# ══════════════════════════════════════════════════════════════
print("\n── Block 8: Perfect Foresight Campaign Continuity ──")
pf_sim_df = df_coast[df_coast.index >= sim_start].copy()
pf_campaign = run_perfect_foresight_campaign(pf_sim_df, spec)

check(
    "starting_soc_pct" in pf_campaign.columns,
    "B8.1 — starting_soc_pct present in PF campaign",
    "B8.1 — starting_soc_pct MISSING from PF campaign",
)
check(
    "ending_soc_pct" in pf_campaign.columns,
    "B8.2 — ending_soc_pct present in PF campaign",
    "B8.2 — ending_soc_pct MISSING from PF campaign",
)

pf_continuity_violations = 0
for i in range(len(pf_campaign) - 1):
    pf_end   = pf_campaign.iloc[i]["ending_soc_pct"]
    pf_start = pf_campaign.iloc[i + 1]["starting_soc_pct"]
    if abs(pf_end - pf_start) > 0.01:
        pf_continuity_violations += 1

check(
    pf_continuity_violations == 0,
    "B8.3 — PF campaign SoC continuous across all day transitions",
    f"B8.3 — {pf_continuity_violations} continuity breaks in PF campaign",
)

check(
    abs(pf_campaign.iloc[0]["starting_soc_pct"] - campaign_df.iloc[0]["starting_soc_pct"]) < 0.01,
    f"B8.4 — PF and DAM campaigns start at same SoC (fair comparison): PF={pf_campaign.iloc[0]['starting_soc_pct']:.2f}% DAM={campaign_df.iloc[0]['starting_soc_pct']:.2f}%",
    f"B8.4 — PF and DAM start at different SoC — unfair comparison",
)

# ══════════════════════════════════════════════════════════════
# BLOCK 9 — MANUAL CONTINUITY VERIFICATION
# ══════════════════════════════════════════════════════════════
print("\n── Block 9: Manual Continuity Verification ──")
three_day_fcst = forecast_df[(forecast_df.index >= sim_start) & (forecast_df.index < sim_start + pd.Timedelta(days=3))]
three_day_camp = run_dam_campaign(three_day_fcst, df_coast, spec)

import ast
replay_engine = SoCEngine(spec)
manual_soc_sequence = [replay_engine.soc_mwh]

for _, row in three_day_camp.iterrows():
    day = pd.Timestamp(row["date"]).date()
    actual_day = df_coast[df_coast.index.date == day]
    charge_hours    = ast.literal_eval(row["charge_hours"]) if isinstance(row["charge_hours"], str) else row["charge_hours"]
    discharge_hours = ast.literal_eval(row["discharge_hours"]) if isinstance(row["discharge_hours"], str) else row["discharge_hours"]
    for h in range(24):
        action = DispatchAction(
            charge_mw=spec.power_mw if h in charge_hours else 0.0,
            discharge_mw=spec.power_mw if h in discharge_hours else 0.0,
        )
        replay_engine.step(action)
    manual_soc_sequence.append(replay_engine.soc_mwh)

for i, (_, row) in enumerate(three_day_camp.iterrows()):
    replay_end   = manual_soc_sequence[i + 1]
    campaign_end = row["ending_soc_pct"] * spec.energy_capacity_mwh / 100
    check(
        abs(replay_end - campaign_end) < 1.0,
        f"B9.{i+1} — Day {i+1} manual replay SoC matches campaign: replay={replay_end/spec.energy_capacity_mwh*100:.2f}% campaign={row['ending_soc_pct']:.2f}%",
        f"B9.{i+1} — Day {i+1} SoC mismatch",
    )

# ══════════════════════════════════════════════════════════════
# BLOCK 10 — REGRESSION: PF ALWAYS >= DAM AFTER FIX
# ══════════════════════════════════════════════════════════════
print("\n── Block 10: PF >= DAM After Continuity Fix ──")
analysis_df = compute_forecast_error_analysis(pf_campaign, three_day_camp)

violations = (analysis_df["dam_revenue"] > analysis_df["pf_net_revenue"] + 0.01).sum()
check(
    violations == 0,
    f"B10.1 — PF revenue >= DAM revenue on every day after fix ({len(analysis_df)} days checked)",
    f"B10.1 — {violations} days where DAM > PF after fix — upper bound violated",
)

negative_error_cost = (analysis_df["forecast_error_cost"] < -0.01).sum()
check(
    negative_error_cost == 0,
    "B10.2 — forecast_error_cost >= 0 on every day (PF is always upper bound)",
    f"B10.2 — {negative_error_cost} days with negative error cost — PF upper bound violated",
)

# ══════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("SoC Continuity Test Results")
print("-" * 60)
print(f"  Total tests:     {passed + failed}")
print(f"  Passed:          {passed}")
print(f"  Failed:          {failed}")
print("-" * 60)
print("\n  Key Metrics:")
print(f"  SoC continuity breaks:  {continuity_violations}")
print(f"  Total violations:       {total_violations} ({avg_per_day:.2f}/day)")
pct_zero = (zero_violation_days / days * 100) if days > 0 else 0
print(f"  Zero-violation days:    {zero_violation_days}/{days} ({pct_zero:.1f}%)")
print("-" * 60)
if failed == 0:
    print("All tests passed — SoC continuity fix verified")
elif continuity_violations == 0 and avg_per_day < 1.0:
    print("Core continuity fix working — review remaining failures")
else:
    print("Continuity fix not fully working — check blocks above")
print("=" * 60)
